# Préparation des données – Sinistres automobiles chez ENSAssuRances

ROY Tao

2026-03-29

## Introduction
Ce document présente le processus complet de préparation et de nettoyage des données issues de la base Sinistre.

L’objectif de cette étape est de produire un jeu de données propre, cohérent et exploitable pour les analyses statistiques.

Les principales étapes de préparation sont :

- import et inspection initiale des données

- rennomage des colonnes

- nettoyage du contenu des variables qualitatives

- nettoyage du contenu des variables quantitatives

- Conversion des dates et tests de cohérences

- Détections et suppression des doublons

- analyse et traitement des valeurs manquantes

- imputation de variables critiques

## 1. Chargement des librairies

Les bibliothèques suivantes sont utilisées pour la manipulation, la visualisation et le nettoyage des données :

In [1]:
import pandas as pd
from itables import show

## 2. Importation des données

Les données sont importées depuis un fichier xlsx.

In [2]:
df_brut = pd.read_excel("data/Sinistre.xlsx")

On fait une inspection de la table chargée :

In [3]:
show(df_brut.head(100), 
     scrollX=True, 
     lengthMenu=[5, 10, 25, 50], 
     pageLength=5,
     classes="cell-border stripe")

Loading ITables v2.7.3 from the internet... (need help?)


In [4]:
print(f"Dimensions : {df_brut.shape}") # (Lignes, Colonnes)
print("-" * 30)
df_brut.info()
display(df_brut.head(3).T) # Aperçu transposé des 3 premières lignes

Dimensions : (72130, 8)
------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 72130 entries, 0 to 72129
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   idx_sin   72130 non-null  str    
 1   gar_sin   72130 non-null  str    
 2   surv_sin  72130 non-null  str    
 3   decl_sin  72130 non-null  str    
 4   clo_sin   33031 non-null  str    
 5   gest_sin  72130 non-null  str    
 6   mt_eval   72130 non-null  float64
 7   mt_regl   72130 non-null  float64
dtypes: float64(2), str(6)
memory usage: 4.4 MB


,0,1,2
idx_sin,S18-0043146,S18-0043146,S18-0043146
gar_sin,AssVHR,AssVHR,AssVHR
surv_sin,2018-12-08,2018-12-08,2018-12-08
decl_sin,2018-12-10,2018-12-10,2018-12-10
clo_sin,NaN,NaN,2019-08-05
gest_sin,2018-12-10,2018-12-31,2019-08-05
mt_eval,170.0,170.0,262.86
mt_regl,0.0,0.0,262.86


## 3. Rennomage des colonnes

Pour une utilisation plus simple de la donnée on va modifier le nom des colonnes comme ceci :

In [6]:
# Création du dictionnaire pour la table Sinistres (8 variables)
data_sinistres_complet = {
    "Nom d'origine": ["idx sin", "gar_sin", "surv sin", "decl sin", "clo sin", "gest_sin", "mt_eval", "mt_regl"],
    "Nouveau nom": [
        "id_sinistre", "type_garantie_sinistree", "date_survenance", "date_declaration", 
        "date_cloture", "date_gestion", "montant_evaluation", "montant_regle"
    ],
    "Type": ["texte", "texte", "texte", "texte", "texte", "texte", "decimal", "decimal"],
    "Description": [
        "Identifiant sinistre", 
        "Garantie sinistrée (Base, 0km, VHR)", 
        "Date de survenance", 
        "Date de déclaration", 
        "Date de clôture", 
        "Date de gestion", 
        "Montant évaluation", 
        "Montant réglé"
    ]
}

df_dico_sinistres = pd.DataFrame(data_sinistres_complet)

print("DICTIONNAIRE DES DONNÉES - SINISTRES")
display(df_dico_sinistres)

DICTIONNAIRE DES DONNÉES - SINISTRES


,Nom d'origine,Nouveau nom,Type,Description
0,idx sin,id_sinistre,texte,Identifiant sinistre
1,gar_sin,type_garantie_sinistree,texte,"Garantie sinistrée (Base, 0km, VHR)"
2,surv sin,date_survenance,texte,Date de survenance
3,decl sin,date_declaration,texte,Date de déclaration
4,clo sin,date_cloture,texte,Date de clôture
5,gest_sin,date_gestion,texte,Date de gestion
6,mt_eval,montant_evaluation,decimal,Montant évaluation
7,mt_regl,montant_regle,decimal,Montant réglé


In [10]:
# Dictionnaire de renommage pour la table Sinistres
mapping_sinistres = {
    "idx sin": "id_sinistre",
    "gar_sin": "type_garantie_sinistree",
    "surv sin": "date_survenance",
    "decl sin": "date_declaration",
    "clo sin": "date_cloture",
    "gest_sin": "date_gestion",
    "mt_eval": "montant_evaluation",
    "mt_regl": "montant_regle"
}

# Application du renommage (remplace df_sinistres par ton nom de variable)
df_prep = df_brut.rename(columns=mapping_sinistres)

Nouveaux noms des 8 colonnes :

In [11]:
df_prep.columns

Index(['idx_sin', 'type_garantie_sinistree', 'surv_sin', 'decl_sin', 'clo_sin',
       'date_gestion', 'montant_evaluation', 'montant_regle'],
      dtype='str')

## 4. Nettoyage du contenu des variables qualitatives

### 4.1 Normalisation de la colonne type_garantie_sinistree

In [12]:
df_prep['type_garantie_sinistree'].value_counts()

type_garantie_sinistree
AssBase    57685
Ass0km      8575
AssVHR      5870
Name: count, dtype: int64

On voit que l'on a bien nos 3 catégories cependant on va changer le nom des valeurs pour qu'elles soient plus explicite :

In [15]:
# Définition de la correspondance (Mapping)
# Basé sur ton dictionnaire : AssBase, Assokm, AssVHR
mapping_garanties = {
    "AssBase": "Assistance de base",
    "Ass0km": "Assistance 0 km",
    "AssVHR": "Véhicule de remplacement"
}

# Application de la transformation
df_prep['type_garantie_sinistree'] = df_prep['type_garantie_sinistree'].replace(mapping_garanties)

# Vérification du nouveau "table()"
print(df_prep['type_garantie_sinistree'].value_counts())

type_garantie_sinistree
Assistance de base          57685
Assistance 0 km              8575
Véhicule de remplacement     5870
Name: count, dtype: int64


## 5. Nettoyage du contenu des variables quantitatives